# Agentic RAG with OpenAI function calling

In this notebook, we will:

1. demonstrate a limitation of direct RAG retrieval;
2. expose the FAQ search function as a tool;
3. let the model formulate a search query;
4. execute the tool in Python;
5. return the result to the model; and
6. calculate the API cost of the complete interaction.

## 1. Set up the OpenAI client

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

## 2. Build the FAQ index

In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

## 3. Test the existing RAG pipeline

First, create the course teaching assistant used in the earlier RAG examples.

In [ ]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

Ask the same question twice: first with the correct spelling of **Ollama**, and then with a deliberate typo.

In [ ]:
print(assistant.rag("How do I run Ollama locally?"))

In [ ]:
print(assistant.rag("How do I run Olama locally?"))

### Limitation: retrieval can be sensitive to spelling

The misspelled query retrieves less relevant context. This illustrates that the retrieval stage may be sensitive to spelling and query phrasing, even though the language model itself could understand the intended question.

## 4. Let the model formulate the search query

Instead of passing the user's text directly to the index, we can expose the search function as a tool. The model can then formulate a suitable query, request the tool, inspect its result, and produce a grounded answer.

Before adding the tool, send the enrollment question to the model without FAQ context to establish a baseline.

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

Without FAQ context, the model can only give a general answer such as "it depends on the course" or "check the course website." It does not know the course-specific enrollment and certificate rules, which is why we need retrieval.

### Define the application-owned search function

The following function contains the retrieval logic used by `RAGBase`. It searches the index directly and returns the five most relevant LLM Zoomcamp FAQ entries.

In [ ]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

### Describe the function to the model

The model does not see the Python implementation. Instead, we provide a tool definition describing what the function does and which arguments it accepts.

The tool definition is independent of the language used to implement the function. The same JSON Schema can describe a tool implemented in Python, TypeScript, Java, or another language.

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The tool name, description, and parameter schema jointly tell the model when and how to use the function. Clear descriptions improve tool selection, while the JSON Schema constrains the expected arguments. Here, `query` is required and additional properties are not allowed.

### Request a tool-assisted answer

Send the same question again, this time including the search tool in the request.

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

Instead of a final answer, the response contains an item whose type is `function_call`. This is a structured request for our application to run the search function.

The generated arguments contain a search query related to late enrollment. Your exact query may differ because model output is not deterministic.

Notice that the model does not necessarily copy the user's question verbatim. It can reformulate the question into terms that may work better for retrieval.

Inspect the generated tool call, its JSON-encoded arguments, and the `call_id` that will connect the search result to this request.

In [ ]:
response.output[0]

In [ ]:
response.output[0].arguments

In [ ]:
response.output[0].call_id

`status="completed"` means the model finished generating the tool-call request. It does **not** mean that the search function has run.

The model does not execute this Python function. It only asks our application to execute it.

## 5. Execute the function and return its result

Our application must:

1. parse the generated arguments;
2. call the corresponding Python function;
3. serialize its result; and
4. return that result using the same `call_id`.

In [ ]:
import json

call = response.output[0]

# Function arguments are returned as a JSON-encoded string.
args = json.loads(call.arguments)

# Custom functions run in our application, not inside the OpenAI API.
# **args unpacks the generated arguments and passes them to search().
results = search(**args)

# Function outputs sent back to the model should be serialized.
result_json = json.dumps(results, indent=2)

In [ ]:
result_json

### Option A: continue with `previous_response_id`

A new API request must be connected to the previous interaction. The simplest approach is to pass `previous_response_id` and provide the function result. The `call_id` tells the model which tool request this result answers.

In [ ]:
final_response = openai_client.responses.create(
    model="gpt-5.4-mini",
    previous_response_id=response.id,
    tools=[search_tool],
    input=[
        {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": result_json,
        }
    ],
)

print(final_response.output_text)

### Option B: manage and replay the history manually

Alternatively, the application can manage the conversation history itself. Add the model's output items to the history, followed by the corresponding function result.

In [ ]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    # Link this result to the model's function-call request.
    "call_id": call.call_id,
    "output": result_json,
})

The model can now see both what it requested and what the function returned. `extend()` adds every item from the `response.output` list, while `append()` adds the single function-result item.

In [ ]:
messages

Replay the complete history to obtain the final answer:

In [ ]:
manual_final_response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(manual_final_response.output_text)

The model now has the original question, its own request to call `search`, and the FAQ results. It can therefore produce a course-specific answer.

Each new request must be connected to the previous interaction. We can either pass `previous_response_id`, as in Option A, or replay the relevant history ourselves, as in Option B.

This is the complete function-calling loop for a single tool call. Compared with a one-call RAG pipeline, the model-driven tool loop requires at least two model requests: one to request the tool and another to interpret its result.

This pattern is commonly described as **agentic RAG**, **tool use**, or **function calling**. The defining feature is that the model decides whether and how to request an available tool.

> **Production note:** This notebook assumes that the response contains exactly one tool call. Production code should inspect every item in `response.output`, execute each `function_call`, and continue the loop until the model returns a final message rather than another tool request.

## 6. Token usage and cost

Option A made two API calls: `response` generated the tool request, and `final_response` interpreted the search result. To calculate the cost of the complete tool-assisted turn, add the usage from both responses.

The manually replayed alternative makes another final-answer request when executed, so treat it as a separate demonstration rather than part of the Option A total.

In [ ]:
responses = [response, final_response]

input_tokens = sum(item.usage.input_tokens for item in responses)
cached_input_tokens = sum(
    item.usage.input_tokens_details.cached_tokens for item in responses
)
output_tokens = sum(item.usage.output_tokens for item in responses)

input_tokens, cached_input_tokens, output_tokens

In [ ]:
# Prices per million tokens for gpt-5.4-mini.
# Verify current prices before using this calculation in production.
def calculate_gpt54mini_price(input_tokens, cached_input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.75
    CACHED_INPUT_PRICE_PER_MILLION = 0.075
    OUTPUT_PRICE_PER_MILLION = 4.50

    uncached_input_tokens = input_tokens - cached_input_tokens
    input_cost = (uncached_input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    cached_input_cost = (
        cached_input_tokens / 1_000_000
    ) * CACHED_INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + cached_input_cost + output_cost

    return {
        "input_cost": input_cost,
        "cached_input_cost": cached_input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

cost = calculate_gpt54mini_price(
    input_tokens=input_tokens,
    cached_input_tokens=cached_input_tokens,
    output_tokens=output_tokens,
)
print("Total cost: $", round(cost["total_cost"], 8))